In [72]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [73]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/weatherAUS_temp3pm_regression.csv')

In [74]:
"""
Linear regression: predict the 3 pm temperature (Temp3pm) in the Australian weather data,
using only information available by 9 am.

Input file : weatherAUS_temp3pm_regression.csv (made from weatherAUS.csv)
             Columns: Date, Split (train/test, by date), Temp9am, MinTemp, Humidity9am,
             Pressure9am, Wind9am_E, Wind9am_N, Month_sin, Month_cos, 48 Loc_* dummies,
             and the target Temp3pm.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------------------------------------------------------------------
# 1. Load data  (in Colab use: "/content/drive/MyDrive/weatherAUS_temp3pm_regression.csv")
# ----------------------------------------------------------------------------
CSV_PATH = "/content/drive/MyDrive/weatherAUS_temp3pm_regression.csv"
df = pd.read_csv(CSV_PATH)
TARGET = "Temp3pm"

# Date and Split are helper columns, not model inputs
FEATURES = [c for c in df.columns if c not in ("Date", "Split", TARGET)]
CONTINUOUS = ["Temp9am", "MinTemp", "Humidity9am", "Pressure9am",
              "Wind9am_E", "Wind9am_N", "Month_sin", "Month_cos"]
LOCATION_COLS = [c for c in FEATURES if c.startswith("Loc_")]

# ----------------------------------------------------------------------------
# 2. Time-based train/test split (train = earlier dates, test = later dates)
# ----------------------------------------------------------------------------
train, test = df[df["Split"] == "train"], df[df["Split"] == "test"]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Train: {len(train):,} rows ({train['Date'].min()} to {train['Date'].max()})")
print(f"Test : {len(test):,} rows ({test['Date'].min()} to {test['Date'].max()})")

# ----------------------------------------------------------------------------
# 3. Fit the model
#    No scaling needed: it does not change linear-regression predictions, and leaving
#    the data unscaled keeps every coefficient in real units (degrees C per unit).
# ----------------------------------------------------------------------------
model = LinearRegression()
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# ----------------------------------------------------------------------------
# 4. Evaluation
# ----------------------------------------------------------------------------
def metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}

results = pd.DataFrame({"train": metrics(y_train, train_pred),
                        "test": metrics(y_test, test_pred)}).round(4)
print("\n=== Model performance ===")
print(results)

# Baseline to beat: "3 pm temp = 9 am temp + average train difference"
avg_gain = (y_train - X_train["Temp9am"]).mean()
baseline_pred = X_test["Temp9am"] + avg_gain
print(f"\nBaseline (Temp9am + {avg_gain:.2f}) test RMSE: "
      f"{np.sqrt(mean_squared_error(y_test, baseline_pred)):.3f}  "
      f"| model test RMSE: {results.loc['RMSE', 'test']:.3f}")

# ----------------------------------------------------------------------------
# 5. Coefficients (real units: degrees C of 3 pm temperature per 1 unit of the feature)
# ----------------------------------------------------------------------------
coefs = pd.Series(model.coef_, index=FEATURES)
print(f"\nIntercept: {model.intercept_:.3f}")
print("\n=== Continuous feature coefficients ===")
print(coefs[CONTINUOUS].round(4).to_string())

loc = coefs[LOCATION_COLS].rename(lambda s: s.replace("Loc_", ""))
print("\n=== Location offsets vs Adelaide (degrees C) ===")
print("Warmest 5:\n", loc.sort_values(ascending=False).head(5).round(2).to_string())
print("Coolest 5:\n", loc.sort_values().head(5).round(2).to_string())

# ----------------------------------------------------------------------------
# 6. Residual diagnostics
# ----------------------------------------------------------------------------
residuals = y_test.to_numpy() - test_pred
print("\n=== Test residuals ===")
print(f"Mean: {residuals.mean():.3f}   Std: {residuals.std():.3f}   "
      f"90% of errors within +/- {np.percentile(np.abs(residuals), 90):.2f} C")

try:
    import matplotlib
    matplotlib.use("Agg")  # remove this line in Colab/Jupyter to show plots inline
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    ax[0].scatter(y_test, test_pred, s=2, alpha=0.2)
    lims = [y_test.min(), y_test.max()]
    ax[0].plot(lims, lims, "r--", lw=1)
    ax[0].set(xlabel="Actual Temp3pm", ylabel="Predicted Temp3pm", title="Actual vs predicted")

    ax[1].scatter(test_pred, residuals, s=2, alpha=0.2)
    ax[1].axhline(0, color="r", lw=1)
    ax[1].set(xlabel="Predicted Temp3pm", ylabel="Residual", title="Residuals vs predicted")

    ax[2].hist(residuals, bins=60)
    ax[2].set(xlabel="Residual (C)", title="Residual distribution")

    plt.tight_layout()
    plt.savefig("regression_diagnostics.png", dpi=130)
    print("\nSaved plots to regression_diagnostics.png")
except ImportError:
    print("\n(matplotlib not installed - skipped plots)")

# ----------------------------------------------------------------------------
# 7. Example: predict for one new morning
#    Use the same feature columns; unlisted Loc_* columns are 0 (Adelaide is the baseline).
# ----------------------------------------------------------------------------
def predict_3pm(temp9am, min_temp, humidity9am, pressure9am,
                wind_speed9am, wind_dir_deg, month, location="Adelaide"):
    theta = np.deg2rad(wind_dir_deg)               # 0 = N, 90 = E, 180 = S, 270 = W
    row = dict.fromkeys(FEATURES, 0)
    row.update({"Temp9am": temp9am, "MinTemp": min_temp, "Humidity9am": humidity9am,
                "Pressure9am": pressure9am,
                "Wind9am_E": wind_speed9am * np.sin(theta),
                "Wind9am_N": wind_speed9am * np.cos(theta),
                "Month_sin": np.sin(2 * np.pi * month / 12),
                "Month_cos": np.cos(2 * np.pi * month / 12)})
    if location != "Adelaide":
        row[f"Loc_{location}"] = 1
    return float(model.predict(pd.DataFrame([row])[FEATURES])[0])

print("\nExample: Sydney, January, 9 am 24 C, min 19 C, humidity 65%, 1015 hPa, 15 km/h from SE")
print(f"Predicted 3 pm temperature: {predict_3pm(24, 19, 65, 1015, 15, 135, 1, 'Sydney'):.1f} C")

Train: 110,178 rows (2007-11-01 to 2015-10-14)
Test : 27,589 rows (2015-10-15 to 2017-06-25)

=== Model performance ===
       train    test
R2    0.8385  0.8414
MAE   2.1016  2.1453
RMSE  2.7578  2.8255

Baseline (Temp9am + 4.75) test RMSE: 3.664  | model test RMSE: 2.825

Intercept: -58.421

=== Continuous feature coefficients ===
Temp9am        1.0031
MinTemp       -0.2633
Humidity9am   -0.0151
Pressure9am    0.0659
Wind9am_E      0.0400
Wind9am_N      0.0350
Month_sin      1.2816
Month_cos      1.1745

=== Location offsets vs Adelaide (degrees C) ===
Warmest 5:
 Katherine    5.20
Uluru        3.43
Mildura      2.84
Woomera      2.70
Darwin       2.64
Coolest 5:
 MountGinini     -2.91
NorfolkIsland   -2.57
Albany          -2.02
Walpole         -1.83
Wollongong      -1.80

=== Test residuals ===
Mean: 0.040   Std: 2.825   90% of errors within +/- 4.47 C

Saved plots to regression_diagnostics.png

Example: Sydney, January, 9 am 24 C, min 19 C, humidity 65%, 1015 hPa, 15 km/h from SE
P